# Aufgabe 12.4: Klassifikationsmodell für defekte Flaschen (Drop Vibration)
Ziel: Vorhersage defekter Flaschen (`is_cracked`) basierend auf der Zeitreihe der Vibrationen (`drop_oscillation`).

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

# Datensatz laden
df = pd.read_csv('data.csv')

In [ ]:
# Relevante Events für Oszillation und Ground Truth extrahieren
df_osc = df[df['event_type'] == 'drop_oscillation'][['bottle', 'drop_oscillation']]
df_truth = df[df['event_type'] == 'ground_truth'][['bottle', 'is_cracked']]

# Über die Flaschen-ID (bottle) zusammenführen und fehlerhafte Zeilen (NaNs) entfernen
df_merged = pd.merge(df_osc, df_truth, on='bottle', how='inner').dropna()

# Zielklasse in Integer konvertieren
y = df_merged['is_cracked'].astype(int)

# Die 500 Messwerte liegen als JSON-String vor. Diese werden in Arrays aus Fließkommazahlen geparst.
df_merged['drop_oscillation'] = df_merged['drop_oscillation'].apply(lambda x: [float(i) for i in json.loads(x)])

In [ ]:
# --- Feature Engineering ---

# 1. Statistische Merkmale (Zeitbereich)
df_merged['mean'] = df_merged['drop_oscillation'].apply(np.mean)
df_merged['std']  = df_merged['drop_oscillation'].apply(np.std)
df_merged['min']  = df_merged['drop_oscillation'].apply(np.min)
df_merged['max']  = df_merged['drop_oscillation'].apply(np.max)
df_merged['median'] = df_merged['drop_oscillation'].apply(np.median)

# 2. Frequenzmerkmale (Frequenzbereich mittels Fast Fourier Transformation - FFT)
# Gemäß den Skript-Theoriegrundlagen ("XFourier, 50Hz ...") extrahieren wir die ersten dominanten Frequenzen.
def get_fft(x, idx): 
    return np.abs(np.fft.fft(x))[idx]

for i in range(1, 6): # Erste 5 Frequenzen
    df_merged[f'fft_{i}'] = df_merged['drop_oscillation'].apply(lambda x: get_fft(x, i))

In [ ]:
# Definition verschiedener Feature-Kombinationen für die geforderte F1-Tabelle
feature_combinations = {
    'mean()': ['mean'],
    'mean(), std()': ['mean', 'std'],
    'mean(), std(), min(), max()': ['mean', 'std', 'min', 'max'],
    'FFT (Freq 1-5)': [f'fft_{i}' for i in range(1, 6)],
    'Statistik + FFT (Freq 1-5)': ['mean', 'std', 'min', 'max', 'median'] + [f'fft_{i}' for i in range(1, 6)]
}

# Modellauswahl (Gewichtung 'balanced' eingefügt, da deutlich mehr intakte als defekte Flaschen existieren)
models = {
    'kNN': KNeighborsClassifier(n_neighbors=5),
    'Log. Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
}

results = []
best_cm = None

# Modelle über alle Feature-Sets iterieren und trainieren
for feat_name, cols in feature_combinations.items():
    X = df_merged[cols]
    
    # Stratifizierter Train-Test-Split (80% Training, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Berechnung der F1-Scores
        f1_train = f1_score(y_train, y_train_pred)
        f1_test = f1_score(y_test, y_test_pred)
        
        results.append({
            'Genutzte Features': feat_name,
            'Modell-Typ': model_name,
            'F1-Score (Training)': f1_train,
            'F1-Score (Test)': f1_test
        })
        
        # Confusion Matrix für das stärkste Modell speichern
        if feat_name == 'Statistik + FFT (Freq 1-5)' and model_name == 'Random Forest':
            best_cm = confusion_matrix(y_test, y_test_pred)

# 1. Ausgabe: F1-Tabelle
results_df = pd.DataFrame(results)
print("--- F1-TABELLE DER KLASSIFIKATIONSMODELLE ---\n")
print(results_df.to_markdown(index=False, floatfmt=".4f"))

# 2. Ausgabe: Confusion Matrix
print("\n\n--- CONFUSION MATRIX ---")
print("Bestes Modell: Random Forest mit allen Features (Statistik + FFT)")
print("(Zeilen: Tatsächliche Klasse [0, 1] | Spalten: Vorhergesagte Klasse [0, 1])\n")
print(best_cm)

--- F1-TABELLE DER KLASSIFIKATIONSMODELLE ---

| Genutzte Features           | Modell-Typ      |   F1-Score (Training) |   F1-Score (Test) |
|:----------------------------|:----------------|----------------------:|------------------:|
| mean()                      | kNN             |                0.1754 |            0.0000 |
| mean()                      | Log. Regression |                0.2190 |            0.1176 |
| mean()                      | Random Forest   |                1.0000 |            0.0870 |
| mean(), std()               | kNN             |                0.7600 |            0.5714 |
| mean(), std()               | Log. Regression |                0.1875 |            0.1724 |
| mean(), std()               | Random Forest   |                1.0000 |            0.6667 |
| mean(), std(), min(), max() | kNN             |                0.5556 |            0.1429 |
| mean(), std(), min(), max() | Log. Regression |                0.1943 |            0.1667 |
| mean(), std